# 12 · Skill evidence, résumé bullets, and interview defense

Map every official node to verifiable evidence and practice making claims no stronger than the results.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Coverage rule

The official page contains 114 active nodes: 84 under AI and 30 under software engineering. Closely
related labels are normalized for implementation, but the official wording is retained for audit.

In [ ]:
skills = pd.read_csv("data/official/ai_rookie_active_skills.csv")
display(skills.groupby(skills.path.str.split("/").str[1]).size().rename("active_nodes"))

## Rebuild the evidence matrix

In [ ]:
subprocess.run([sys.executable, "scripts/build_skill_matrix.py"], check=True)
matrix_text = Path("docs/official-skill-matrix.md").read_text()
print("matrix rows", matrix_text.count("\n|"), "characters", len(matrix_text))
print("\n".join(matrix_text.splitlines()[:12]))

## Sample an interview trail

In [ ]:
terms = ["CNN", "MySQL", "BERT", "GCN", "PyTorch", "precision"]
rows = [line for line in matrix_text.splitlines()
        if any(term.lower() in line.lower() for term in terms)]
display(pd.Series(rows[:20], name="evidence trail").to_frame())

## Result-aware résumé writing

Use: action + technical scope + evaluation + result + limitation. Never say “production-grade” from
a localhost API, “real customer impact” from synthetic data, or “state of the art” from a toy model.

In [ ]:
classical = json.loads(Path("artifacts/metrics/classical.json").read_text())
best_name, best = max(classical["classification"].items(), key=lambda item: item[1]["roc_auc"])
bullet = (f"Built a leakage-safe {best_name} churn benchmark across a multi-model suite; "
          f"measured ROC-AUC {best['roc_auc']:.3f}, PR-AUC {best['pr_auc']:.3f}, "
          "calibration, errors, and CPU latency on versioned synthetic retail data.")
print(bullet)

## Defend negative results

In [ ]:
graph = json.loads(Path("artifacts/metrics/graph.json").read_text())
display(pd.Series({"GCN link ROC-AUC": graph["link_prediction"]["gcn_roc_auc"],
                   "degree baseline": graph["link_prediction"]["degree_popularity_baseline_roc_auc"]}))

A negative comparison is useful when the protocol is fair and preserved. Explain the likely
causes, propose the next discriminating experiment, and avoid retroactively changing the metric.

In [ ]:
print(Path("docs/resume-guide.md").read_text())
print(Path("docs/interview-guide.md").read_text())

## Mock interview prompts

1. Where can leakage enter the classical pipeline?
2. Why can high accuracy coexist with poor recall?
3. Why is the graph evaluation transductive?
4. What does the transfer-learning proxy fail to prove?
5. Which API controls are still missing?
6. Which single experiment would most strengthen this portfolio, and why?